In [ ]:
# 💡 코딩 튜터의 친절한 한마디:
# "안녕하세요! 오늘은 음성 인식(ASR)이라는 정말 멋진 분야에 도전할 거예요.
# 실제 오디오 파일을 다루는 건 복잡할 수 있지만, 우리는 이 데이터가
# 어떤 구조로 되어 있는지, 그리고 어떤 정보를 담고 있는지 분석하는
# 재미있는 데이터 탐험을 해볼 거예요. 걱정 마세요, 차근차근 따라오면 금방 실력자가 될 거예요!"

# 📜 데이터셋 정보:
# 제목: Korean Speech Dataset (한국어 음성 데이터셋)
# 의미: 20명 이상의 한국어 화자가 녹음한 10시간 이상의 음성 녹음 자료입니다.
# 설명: 주로 자동 음성 인식(ASR) 모델을 학습시키거나, 한국어 자연어 처리(NLP) 모델을 개발하는 데 사용됩니다.
# 우리는 이 메타데이터를 분석하여 '어떤 종류의 데이터가 얼마나 많은지'를 알아보며 AI의 기초를 다질 거예요!

import random
from tqdm import tqdm
from datasets import load_dataset, get_dataset_config_names
import numpy as np
import pandas as pd

# =============================================================================
# ⚙️ 환경 설정 및 상수 정의
# =============================================================================
DATASET_NAME = "UniDataPro/korean-speech-recognition"
SAMPLE_COUNT = 10 # 테스트를 위해 상위 10개 샘플만 사용합니다.

# 🚀 데이터셋의 Config 정보를 먼저 확인해 봅시다!
print("=" * 80)
print("🌐 [Step 1/4] 데이터셋 Config 정보 로드 및 확인")
print("=" * 80)

try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 첫 번째 Config로 로드합니다.
    selected_config = configs[0]
    print(f"👉 다음부터 {selected_config} 설정을 사용하여 데이터를 로드합니다.")

except Exception as e:
    print(f"ℹ️ Config 로드 중 오류 발생: {e}. 기본(default) 설정을 사용합니다.")
    selected_config = None

# =============================================================================
# 💾 데이터 로드 및 스트리밍 전략 (★ 필수 패턴 적용!)
# =============================================================================

dataset = None
try:
    # 1. 스트리밍 모드를 시도합니다. (대용량 데이터 처리에 최적!)
    print("\n🌟 [Step 2/4] 스트리밍 모드(streaming=True)로 데이터 로드를 시도합니다...")
    dataset = load_dataset(DATASET_NAME, name=selected_config, split='train', streaming=True)
    print("🎉 스트리밍 로드 성공! 데이터셋을 메모리에 모두 올리지 않고 효율적으로 처리할 수 있어요.")
except Exception as e:
    # 스트리밍 로드가 실패할 경우 (예: 인터넷 연결 문제, 구조적 문제)
    print(f"⚠️ 스트리밍 로드 실패 ({e}). 일반 Dataset 형태로 소량 다운로드하여 진행합니다.")
    try:
        # 2. 일반 Dataset으로 소량 로드합니다.
        dataset = load_dataset(DATASET_NAME, name=selected_config, split='train', streaming=False)
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드에 실패했습니다. 오류: {e_fallback}")
        exit()

# 📥 샘플링 로직: Streaming 또는 Standard Dataset에 맞게 처리합니다.
print("\n✨ [Step 3/4] 데이터 샘플링 및 반복자(Iterator) 설정")
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)일 가능성이 높습니다.
    # -> take(K)로 K개만 가져오고, 이를 iterator로 만듭니다.
    print(f"   [Mode] 스트리밍 모드 감지: 상위 {SAMPLE_COUNT}개만 가져옵니다.")
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
else:
    # 일반 Dataset인 경우
    print(f"   [Mode] 일반 Dataset 감지: 상위 {SAMPLE_COUNT}개만 메모리에 로드합니다.")
    sampled_dataset = list(dataset.take(SAMPLE_COUNT))
    # 목록 형태로 로드되었으므로, list 자체를 반복자로 사용합니다.
    sampled_dataset_iterator = iter(sampled_dataset)


# =============================================================================
# 🔬 데이터 탐험 및 창의적 실습 (Audio Corpus Profiler)
# =============================================================================

print("\n=============================================================================")
print("🚀 [Step 4/4] ✨ 초급 AI 실습: 음성 코퍼스 메타데이터 프로파일러")
print("=============================================================================")

# 💡 목표: 데이터셋이 담고 있는 '다양성'을 분석합니다.
# 실제 오디오 파일을 다루지 않고, 오직 메타데이터(Language, Format 등)만으로
# 이 데이터셋이 얼마나 풍부한지 통계적으로 확인해봅시다!

# 1. 핵심 메타데이터 수집 및 분석 (정량적 분석 포함)
metadata_list = []
print("✅ 1. 샘플 메타데이터 추출 (최대 10개)")
print("----------------------------------------------")

for i, sample_data in enumerate(tqdm(sampled_dataset_iterator, desc="Reading Samples")):
    # ⚠️ 중요: 반복자(iterator)에서 다음 데이터를 가져올 때는 next()를 사용합니다.
    try:
        sample = next(sampled_dataset_iterator)
        metadata_list.append(sample)
    except StopIteration:
        break

# 2. DataFrame으로 변환하여 보기 쉽게 만듭니다.
df_metadata = pd.DataFrame(metadata_list)

# 3. 데이터 통계 분석 (가장 흥미로운 부분!)
print("\n✨ 2. 데이터셋의 다양성 분석 결과:")
print("-----------------------------------------------------------")

# 🔑 Language 분석: 어떤 언어가 주로 쓰였는지 확인합니다.
unique_languages = df_metadata['Language'].unique()
print(f"   - 🗣️ 언어 다양성 (Language): {len(unique_languages)}개의 고유 언어 발견!")
print(f"     (가장 많이 쓰인 언어: {df_metadata['Language'].mode()[0] if not df_metadata.empty else 'N/A'})")

# 🔑 Format 분석: 녹음 파일이 어떤 형식(Format)으로 되어 있는지 확인합니다.
unique_formats = df_metadata['Format'].unique()
print(f"   - 🗂️ 파일 형식 (Format): {len(unique_formats)}가지 포맷이 존재합니다.")

# 🔑 Minutes 분석: 시간 단위 데이터를 분석합니다.
# (실제로는 이 데이터를 숫자로 변환하여 분석해야 합니다!)
print(f"   - ⏳ 길이 정보 (Minutes): 총 {len(unique_formats)}개의 샘플이 기록된 메타데이터가 있습니다.")

# 4. 데이터 요약 출력
print("\n=============================================================================")
print("🎉 성공! 기본 데이터 탐색 실습을 완료했습니다!")
print("🎉 [튜터 요약]: 우리는 데이터셋의 전반적인 '골격'을 이해했습니다.")
print("     실제 ASR 작업에서는 여기에 오디오 파일 경로와 함께 추출된 특징(Feature)을 분석하게 됩니다.")
print("=============================================================================")